# Самостоятельная работа: dense GAN на Fashion-MNIST

Здесь требуется самостоятельно перенести алгоритм состязательного
обучения на Fashion-MNIST. Данные отличаются от MNIST-примера, но
размер изображения, нормализация, архитектуры и математические цели
остаются теми же. Заголовки описывают этапы; маркеры «TODO(student)»
расположены только в кодовых ячейках.

Выполняйте задания строго по порядку. На каждом этапе сначала
подтвердите форму тензоров и только потом продолжайте. Метки классов
Fashion-MNIST не становятся целями GAN: задача по-прежнему состоит в
различении реального и синтетического изображения.

## История одного пакета

Генератор превращает пачку случайных карточек `(B,100)` в пачку
картинок `(B,784)`. Дискриминатор смотрит и на эти картинки, и на
настоящие предметы одежды, а затем выдаёт по одному числу `(B,1)`.
Сначала учится проверяющий, потом — художник. Выполняйте этапы именно
по порядку: следующий этап использует объекты, созданные и проверенные
на предыдущем.

### Маленький словарь перед реализацией

Параметры — обучаемые числа в слоях. Прямой проход создаёт картинку и
ответ проверяющего. Потеря — одно число, описывающее ошибку текущего
ответа. Градиент — инструкция, как слегка поменять параметры, чтобы
ошибка уменьшилась. `GradientTape` находит такую инструкцию, а Adam
применяет небольшую поправку.

## Этап 1. Конфигурация независимого эксперимента

**Доступно:** константы ниже. **Цель:** зафиксировать Fashion-MNIST,
первые 10 000 изображений, $d_z=100$, $d_x=784$, пакет 128 и
первоначальные 105 эпох. Для сравнений позднее установите 205 и 400
эпох, не меняя остальную постановку. **Контрольная точка:** диапазон
целевых меток GAN — 0.9, 0 и 1; это не метки предметов одежды.

`(B,784)` означает «B картинок, каждая записана длинной строкой из
784 яркостей», а `(B,1)` — один ответ проверяющего на картинку. Это не
784 разных объектов и не десять классов Fashion-MNIST.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf


SEED = 42
DATASET_NAME = 'Fashion-MNIST'
MAX_EXAMPLES = 10_000
IMAGE_HEIGHT = 28
IMAGE_WIDTH = 28
IMAGE_DIM = IMAGE_HEIGHT * IMAGE_WIDTH   # 784
NOISE_DIM = 100                          # d_z
BATCH_SIZE = 128
EPOCHS = 105
LEARNING_RATE = 2e-4
BETA_1 = 0.5
REAL_LABEL = 0.9
FAKE_LABEL = 0.0
GENERATOR_TARGET = 1.0
SAVE_EVERY = 20

# seed ДО создания слоёв и первого шума — иначе прогоны несравнимы
tf.keras.utils.set_random_seed(SEED)

artifact_dir = Path('artifacts/fashion_mnist_dense_gan')
artifact_dir.mkdir(parents=True, exist_ok=True)


def load_fashion_mnist(max_examples: int = MAX_EXAMPLES):
    (x_train, _), (x_test, _) = tf.keras.datasets.fashion_mnist.load_data()

    # «первые 10 000 изображений» — берём из train, как есть
    x = x_train[:max_examples]

    # (N, 28, 28) uint8 -> (N, 784) float32 в диапазоне [-1, 1]
    x = x.reshape(-1, IMAGE_DIM).astype('float32')
    x = (x - 127.5) / 127.5

    return x


def make_dataset(x: np.ndarray, batch_size: int = BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices(x)
    ds = ds.shuffle(buffer_size=len(x), seed=SEED)
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


if __name__ == '__main__':
    x = load_fashion_mnist()
    ds = make_dataset(x)

    print(f'{DATASET_NAME}: x.shape = {x.shape}, dtype = {x.dtype}')
    print(f'range: [{x.min():.2f}, {x.max():.2f}]')

    for batch in ds.take(1):
        print(f'batch shape = {batch.shape}  (B, 784)')  # (128, 784)

    print(f'steps per epoch = {len(x) // BATCH_SIZE}')     # 78
    print(f'artifact dir     = {artifact_dir.resolve()}')

: 

## Этап 2. Получение и подготовка Fashion-MNIST

**Требуется:** получить обучающие изображения Fashion-MNIST, оставить
первые 10 000, преобразовать пиксели из $[0,255]$ в $[-1,1]$ и
развернуть их в $(N,784)$. Затем создайте перемешиваемый набор
мини-пакетов $(B,784)$ без меток классов. **Проверьте:** значения
конечны и принадлежат $[-1,1]$; метки используются только для
необязательной подписи исходных изображений.

**Маршрут:** сначала загрузите массив и посмотрите на одну картинку,
затем приведите значения к `float32`, измените шкалу `[0,255] → [-1,1]`
и лишь потом разверните карту. Останавливайтесь на `assert`: при
неверной форме или диапазоне сеть строить ещё рано. `y_train` не
передаётся GAN, иначе это была бы задача классификации одежды.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

# ----------------------------------------------------------------------
# Конфигурация (та же, что на Этапе 1)
# ----------------------------------------------------------------------
SEED = 42
MAX_EXAMPLES = 10_000
IMAGE_HEIGHT = 28
IMAGE_WIDTH = 28
IMAGE_DIM = IMAGE_HEIGHT * IMAGE_WIDTH   # 784
BATCH_SIZE = 128

tf.keras.utils.set_random_seed(SEED)
artifact_dir = Path('artifacts/fashion_mnist_dense_gan')
artifact_dir.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------------------
# Шаг 1. Получить x_train (N,28,28) и y_train (N,) — только первые 10 000.
#         y_train НЕ идёт в Dataset (это имя одежды, не ответ GAN).
# ----------------------------------------------------------------------
(x_full, y_full), _ = tf.keras.datasets.fashion_mnist.load_data()

x_train = x_full[:MAX_EXAMPLES]          # (10000, 28, 28) uint8
y_train = y_full[:MAX_EXAMPLES]          # (10000,)         uint8

assert x_train.shape == (MAX_EXAMPLES, IMAGE_HEIGHT, IMAGE_WIDTH), x_train.shape
assert y_train.shape == (MAX_EXAMPLES,), y_train.shape


# ----------------------------------------------------------------------
# Шаг 2. Сначала посмотреть на одну картинку (до любых преобразований).
# ----------------------------------------------------------------------
def show_one(image: np.ndarray, label: int | None = None) -> None:
    plt.figure(figsize=(3, 3))
    plt.imshow(image, cmap='gray')
    plt.axis('off')
    if label is not None:
        plt.title(f'label={label}')
    plt.tight_layout()
    plt.show()


show_one(x_train[0], int(y_train[0]))    # необязательная подпись исходника


# ----------------------------------------------------------------------
# Шаг 3. float32 -> линейно [0,255] в [-1,1] -> развернуть 28×28 в 784.
#         Порядок важен: масштабируем ДО reshape, потом разворачиваем.
# ----------------------------------------------------------------------
x_train = x_train.astype('float32')                    # [0,255], float32
x_train = (x_train / 127.5) - 1.0                      # [0,255] -> [-1,1]
x_train = x_train.reshape(-1, IMAGE_DIM)               # (N,28,28) -> (N,784)

assert x_train.shape == (MAX_EXAMPLES, IMAGE_DIM), x_train.shape
assert np.isfinite(x_train).all(), 'non-finite values in x_train'
assert x_train.min() >= -1.0 and x_train.max() <= 1.0, \
    f'range out of [-1,1]: [{x_train.min()}, {x_train.max()}]'


# ----------------------------------------------------------------------
# Шаг 4. Dataset только из x_train. До batching — (784,), после — (B,784).
#         Метки классов не передаём.
# ----------------------------------------------------------------------
dataset = (
    tf.data.Dataset.from_tensor_slices(x_train)        # элемент: (784,)
    .shuffle(buffer_size=len(x_train), seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE, drop_remainder=True)            # пакет: (B,784)
    .prefetch(tf.data.AUTOTUNE)
)


# ----------------------------------------------------------------------
# Контроль
# ----------------------------------------------------------------------
if __name__ == '__main__':
    print(f'x_train: shape={x_train.shape}, dtype={x_train.dtype}, '
          f'range=[{x_train.min():.2f}, {x_train.max():.2f}]')
    print(f'y_train: shape={y_train.shape}, dtype={y_train.dtype} '
          f'(в Dataset не передаётся)')

    elem = next(iter(tf.data.Dataset.from_tensor_slices(x_train).take(1)))
    assert elem.shape == (IMAGE_DIM,), elem.shape
    print(f'элемент до batching: {elem.shape}  (784,)')

    for batch in dataset.take(1):
        assert batch.shape == (BATCH_SIZE, IMAGE_DIM), batch.shape
        assert np.isfinite(batch.numpy()).all()
        assert batch.numpy().min() >= -1.0 and batch.numpy().max() <= 1.0
        print(f'первый пакет:       {batch.shape}  (B,784)')

    print(f'steps per epoch = {len(x_train) // BATCH_SIZE}')   # 78

## Этап 3. Генератор и проверка его выхода

**Требуется:** собрать генератор
$100\to256\to512\to1024\to784$ с тремя LeakyReLU и финальным tanh.
Вход — шум $(B,100)$ с независимыми нормальными координатами; выход —
синтетические развёрнутые изображения $(B,784)$ в $[-1,1]$.
Используйте современный tf.keras и не применяйте готовый GAN-слой.
**Контрольная точка:** выход имеет точную форму и не содержит NaN или
бесконечностей.

Считайте `z` карточкой со 100 случайными ручками. В начале генератор
рисует бессмыслицу; обучение понемногу поправляет его веса, чтобы те
же 100 чисел превращались в структуру, похожую на данные. `tanh` не
даёт пикселям покинуть диапазон настоящих картинок.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import tensorflow as tf

# ----------------------------------------------------------------------
# Конфигурация (та же, что на Этапах 1–2)
# ----------------------------------------------------------------------
SEED = 42
NOISE_DIM = 100          # d_z
IMAGE_DIM = 28 * 28      # 784
BATCH_SIZE = 128

tf.keras.utils.set_random_seed(SEED)
artifact_dir = Path('artifacts/fashion_mnist_dense_gan')
artifact_dir.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------------------
# Шаг 1. Initializer + фабрика. Фабрика только создаёт θ_G.
#         Input задаёт форму одного z_i = (100,); ось B не пишем.
# ----------------------------------------------------------------------
def make_initializer() -> tf.keras.initializers.Initializer:
    # He-подобный нормальный initializer — стандарт для LeakyReLU-стека.
    return tf.keras.initializers.RandomNormal(
        mean=0.0, stddev=0.02, seed=SEED
    )


def make_generator(
    noise_dim: int = NOISE_DIM,
    image_dim: int = IMAGE_DIM,
) -> tf.keras.Model:
    """G: (B, 100) -> (B, 784), выход в [-1, 1] через tanh."""
    init = make_initializer()

    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(noise_dim,)),          # один z_i: (100,)

            tf.keras.layers.Dense(256, kernel_initializer=init, use_bias=False),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),

            tf.keras.layers.Dense(512, kernel_initializer=init, use_bias=False),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),

            tf.keras.layers.Dense(1024, kernel_initializer=init, use_bias=False),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),

            tf.keras.layers.Dense(image_dim, kernel_initializer=init),
            tf.keras.layers.Activation('tanh'),                 # (B, 784) in [-1, 1]
        ],
        name='generator',
    )


# ----------------------------------------------------------------------
# Шаг 2. Построить G, сформировать z ~ N(0, I) формы (B, 100),
#         вызвать G(z, training=False). Никакого GradientTape/оптимизатора.
# ----------------------------------------------------------------------
if __name__ == '__main__':
    generator = make_generator()
    generator.summary()

    # независимые нормальные координаты, (B, 100)
    z = tf.random.normal(shape=(BATCH_SIZE, NOISE_DIM), mean=0.0, stddev=1.0)

    # диагностический прямой проход — без обучения
    x_fake = generator(z, training=False)

    # контрольная точка: точная форма и отсутствие NaN/inf
    assert x_fake.shape == (BATCH_SIZE, IMAGE_DIM), x_fake.shape
    x_np = x_fake.numpy()
    assert np.isfinite(x_np).all(), 'NaN or inf in G(z)'
    assert x_np.min() >= -1.0 and x_np.max() <= 1.0, \
        f'out of [-1,1]: [{x_np.min()}, {x_np.max()}]'

    print(f'z:      shape={z.shape}, mean={tf.reduce_mean(z):.3f}, '
          f'std={tf.math.reduce_std(z):.3f}')
    print(f'G(z):   shape={x_fake.shape}, dtype={x_fake.dtype}, '
          f'range=[{x_np.min():.3f}, {x_np.max():.3f}]')
    print(f'finite: {np.isfinite(x_np).all()}')

## Этап 4. Дискриминатор и смысл его выхода

**Требуется:** собрать дискриминатор
$784\to1024\to512\to256\to1$ с LeakyReLU, Dropout(0.3) и sigmoid.
Его входом по очереди будут настоящие и синтетические изображения
$(B,784)$, а результатом — $(B,1)$ вероятностей. Последняя координата
— не класс Fashion-MNIST, а оценка «настоящее».
**Контрольная точка:** на диагностическом входе значения строго между
0 и 1.

Единственный вопрос дискриминатора — «эта картинка пришла из набора
или создана генератором?». Его ответ после sigmoid можно читать как
уверенность от 0 до 1. Это совсем не вероятность одного из классов
одежды и не оценка того, насколько красив предмет.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import tensorflow as tf

# ----------------------------------------------------------------------
# Конфигурация (та же, что на Этапах 1–3)
# ----------------------------------------------------------------------
SEED = 42
NOISE_DIM = 100
IMAGE_DIM = 28 * 28
BATCH_SIZE = 128
DROPOUT_RATE = 0.3

tf.keras.utils.set_random_seed(SEED)
artifact_dir = Path('artifacts/fashion_mnist_dense_gan')
artifact_dir.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------------------
# Initializer и фабрики (повторяем G из Этапа 3, добавляем D)
# ----------------------------------------------------------------------
def make_initializer() -> tf.keras.initializers.Initializer:
    return tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02, seed=SEED)


def make_generator(noise_dim: int = NOISE_DIM,
                   image_dim: int = IMAGE_DIM) -> tf.keras.Model:
    init = make_initializer()
    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(noise_dim,)),
            tf.keras.layers.Dense(256, kernel_initializer=init, use_bias=False),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dense(512, kernel_initializer=init, use_bias=False),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dense(1024, kernel_initializer=init, use_bias=False),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dense(image_dim, kernel_initializer=init),
            tf.keras.layers.Activation('tanh'),
        ],
        name='generator',
    )


# ----------------------------------------------------------------------
# Шаг 1. Дискриминатор: 784 -> 1024 -> 512 -> 256 -> 1, LeakyReLU,
#         Dropout(0.3), финальный sigmoid. Возвращает (B,1) — P(настоящее).
# ----------------------------------------------------------------------
def make_discriminator(image_dim: int = IMAGE_DIM,
                       dropout_rate: float = DROPOUT_RATE) -> tf.keras.Model:
    """D: (B, 784) -> (B, 1), вероятности строго в (0, 1)."""
    init = make_initializer()

    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(image_dim,)),           # один x_i: (784,)

            tf.keras.layers.Dense(1024, kernel_initializer=init),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dropout(dropout_rate),               # активен при training=True

            tf.keras.layers.Dense(512, kernel_initializer=init),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dropout(dropout_rate),

            tf.keras.layers.Dense(256, kernel_initializer=init),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dropout(dropout_rate),

            tf.keras.layers.Dense(1, kernel_initializer=init),
            tf.keras.layers.Activation('sigmoid'),               # (B, 1) in (0, 1)
        ],
        name='discriminator',
    )


# ----------------------------------------------------------------------
# Шаг 2. Проверка: пропустить реальный и синтетический пакеты через D.
#         Форма (B,1), конечные значения, строго в (0,1). Не класс одежды.
# ----------------------------------------------------------------------
if __name__ == '__main__':
    generator = make_generator()
    discriminator = make_discriminator()
    discriminator.summary()

    # реальный пакет: сымитируем загрузкой Fashion-MNIST
    (x_full, _), _ = tf.keras.datasets.fashion_mnist.load_data()
    x_real = x_full[:BATCH_SIZE].reshape(-1, IMAGE_DIM).astype('float32')
    x_real = (x_real / 127.5) - 1.0
    x_real = tf.convert_to_tensor(x_real)

    # синтетический пакет
    z = tf.random.normal((BATCH_SIZE, NOISE_DIM))
    x_fake = generator(z, training=False)

    # диагностические проходы через D. training=False — Dropout выключен,
    # чтобы результат был детерминированным для проверки диапазона.
    d_real = discriminator(x_real, training=False)
    d_fake = discriminator(x_fake, training=False)

    # контрольная точка
    for name, d in (('D(x_real)', d_real), ('D(x_fake)', d_fake)):
        assert d.shape == (BATCH_SIZE, 1), (name, d.shape)
        d_np = d.numpy()
        assert np.isfinite(d_np).all(), f'NaN/inf in {name}'
        assert (d_np > 0.0).all() and (d_np < 1.0).all(), \
            f'{name} not strictly in (0,1): [{d_np.min()}, {d_np.max()}]'
        print(f'{name}: shape={d.shape}, '
              f'range=({d_np.min():.6f}, {d_np.max():.6f}), '
              f'mean={d_np.mean():.4f}')

    # Напоминание: последняя координата — P(настоящее), не класс одежды.
    print('\nD отвечает на вопрос «из набора или от G?», а не «какая одежда?».')

## Этап 5. Две функции потерь и два независимых обновления

**Требуется:** создать BinaryCrossentropy с «from_logits=False» и два
Adam-оптимизатора с заданными learning_rate и beta_1. Напишите
попеременный шаг: сначала обновляет только дискриминатор по настоящим
меткам 0.9 и синтетическим 0; затем обновляет только генератор по
цели 1. Для первого шага отделите синтетические изображения от
градиента генератора. Для второго вызовите дискриминатор с
«training=False», но сохраните прохождение градиента к генератору.

**Контрольная точка:** после первого подшага меняются только
переменные дискриминатора, после второго — только переменные
генератора. Обе потери должны быть конечными скалярами.

**Сначала → затем → проверить.** Сначала выполните фазу D и отделите
созданные картинки через `stop_gradient`: обратная стрелка к G
выключена. Затем создайте новый `GradientTape` для фазы G: стрелка от
ошибки идёт через D к G, но список параметров оптимизатора содержит
только G. Визуальная проверка проста: спросите, какие веса стоят справа
в каждом вызове `apply_gradients`.

Здесь D возвращает вероятность $p$ после sigmoid. Для одной цели $y$
API `BinaryCrossentropy(from_logits=False)` использует

$$
\operatorname{BCE}(y,p)=-y\log p-(1-y)\log(1-p).
$$

Неправильный уверенный ответ получает большую потерю; правильный
уверенный ответ — малую. Для всего мини-пакета TensorFlow усредняет
эту ошибку, получая один скаляр для каждого игрока.

### Математический контракт вашей реализации

**Сначала посмотрите на две фазы глазами данных.**

![Два независимых обновления GAN: обучение дискриминатора, затем обучение генератора](assets/diagrams/gan_two_phase_training.svg)

Эта схема разделяет временные тензоры, скалярные потери и параметры.
Ваша будущая функция не должна пытаться «обучать шум» или метки
Fashion-MNIST: шум — вход, а метки классов вообще не участвуют в GAN.
Обучаемыми остаются только `tf.Variable` внутри G и D.

Обозначим веса и смещения генератора через $\theta_G$, а параметры
дискриминатора — через $\theta_D$. Потеря — один скаляр. Производная
по параметру показывает направление его маленькой поправки:

$$
\theta\leftarrow\theta-\eta\nabla_\theta\mathcal L.
$$

В первой фазе для мини-пакета должны быть вычислены два члена
$\operatorname{BCE}(0.9,D(\mathbf x_{\rm real}))$ и
$\operatorname{BCE}(0,D(G(\mathbf z_D)))$; их среднее образует
$\mathcal L_D$, а градиент запрашивается только по $\theta_D$.
Во второй фазе $\mathcal L_G=\operatorname{BCE}(1,D(G(\mathbf z_G)))$
дифференцируется только по $\theta_G$. Шум — не параметр: каждый
$\mathbf z$ является новым входом формы `(B,100)`.

### Как `GradientTape` превращает loss в поправки весов

![Жизненный цикл GradientTape от прямого прохода до обновления Adam](assets/diagrams/tensorflow_gradient_tape.svg)

`GradientTape` записывает прямой путь вычислений до `loss`, а не
делает отдельное обучение. Пока активен блок `with`, он помнит
дифференцируемые операции и автоматически наблюдает используемые
обучаемые `tf.Variable`. Затем `tape.gradient(loss, variables)`
запускает обратный проход и возвращает по одной производной для
каждого элемента `variables`; форма градиента совпадает с формой
веса. `zip` сопоставляет эти два списка, а `apply_gradients` передаёт
пары Adam. Именно `variables` определяет, обновляется $\theta_G$ или
$\theta_D$; имя tape этого решения не принимает.

Непостоянный tape имеет одно назначение: он нужен для одного вызова
`gradient`, затем его запись освобождается. Это программная запись
правила градиентного шага, а не эвристика TensorFlow. Производные
получаются по цепному правилу: библиотека не выписывает гигантские
якобианы, а передаёт их нужное действие от скалярной потери к весам.

### Один мини-пакет: карта объектов вашей реализации

Используйте эти имена как ориентиры при заполнении TODO. Объекты с
пометкой «временный» не являются параметрами и не передаются Adam.

| Математика | Ожидаемое имя | Форма | Роль в одной фазе |
| --- | --- | --- | --- |
| $\mathbf x_{\rm real}$ | `real_images` | `(B,784)`, `float32` | Настоящие картинки для D. |
| $\mathbf z_D$ | `discriminator_noise` | `(B,100)` | Временный шум фазы D. |
| $G(\mathbf z_D)$ | `fake_images` | `(B,784)` | Подделки после `stop_gradient`. |
| $D(\mathbf x)$ | `real_scores`, `fake_scores` | `(B,1)`, вероятность | Аргумент BCE фазы D. |
| $\mathcal L_D$ | `discriminator_loss` | скаляр | Даёт градиенты только для `discriminator.trainable_variables`. |
| $\mathbf z_G$ | `generator_noise` | `(B,100)` | Новый шум, независимый от $\mathbf z_D$. |
| $G(\mathbf z_G)$ | `generated_images` | `(B,784)` | Подделки, оставленные в графе. |
| $\mathcal L_G$ | `generator_loss` | скаляр | Даёт градиенты только для `generator.trainable_variables`. |

**Проведите одну фазу буквально по строкам.** Сначала внутри первого
`GradientTape` вычисляются входы D и один скаляр `discriminator_loss`.
Затем `tape.gradient(discriminator_loss,
discriminator.trainable_variables)` строит список производных в том же
порядке, что список весов. `zip` образует пары, и только после
`discriminator_optimizer.apply_gradients(...)` меняются $\theta_D$.
Фаза G повторяет эту схему с новым tape, новым шумом и списком весов G.

Для этой модели D выдаёт вероятность, поэтому точный путь градиента
генератора можно записать через якобианы:

$$
\nabla_{\theta_G}\mathcal L_G
=J_{G,\theta_G}^{\mathsf T}J_{D,\mathbf x}^{\mathsf T}
\nabla_{\mathbf p}\mathcal L_G.
$$

Якобиан — таблица всех локальных влияний: как выход меняется при малом
изменении входа. Он слишком велик для явного создания; `gradient`
вычисляет только нужный обратный проход, то есть произведения с ним.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import tensorflow as tf

# ----------------------------------------------------------------------
# Конфигурация (та же, что на Этапах 1–4)
# ----------------------------------------------------------------------
SEED = 42
NOISE_DIM = 100
IMAGE_DIM = 28 * 28
BATCH_SIZE = 128
LEARNING_RATE = 2e-4
BETA_1 = 0.5
REAL_LABEL = 0.9
FAKE_LABEL = 0.0
GENERATOR_TARGET = 1.0
DROPOUT_RATE = 0.3

tf.keras.utils.set_random_seed(SEED)
artifact_dir = Path('artifacts/fashion_mnist_dense_gan')
artifact_dir.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------------------
# Модели из Этапов 3–4 (фабрики θ_G и θ_D)
# ----------------------------------------------------------------------
def make_initializer():
    return tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02, seed=SEED)


def make_generator(noise_dim=NOISE_DIM, image_dim=IMAGE_DIM):
    init = make_initializer()
    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(noise_dim,)),
            tf.keras.layers.Dense(256, kernel_initializer=init, use_bias=False),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dense(512, kernel_initializer=init, use_bias=False),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dense(1024, kernel_initializer=init, use_bias=False),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dense(image_dim, kernel_initializer=init),
            tf.keras.layers.Activation('tanh'),
        ],
        name='generator',
    )


def make_discriminator(image_dim=IMAGE_DIM, dropout_rate=DROPOUT_RATE):
    init = make_initializer()
    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(image_dim,)),
            tf.keras.layers.Dense(1024, kernel_initializer=init),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(512, kernel_initializer=init),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(256, kernel_initializer=init),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(1, kernel_initializer=init),
            tf.keras.layers.Activation('sigmoid'),
        ],
        name='discriminator',
    )


generator = make_generator()
discriminator = make_discriminator()


# ----------------------------------------------------------------------
# Шаг 1. BCE + ДВА независимых Adam. Состояния моментов G и D не общие.
#         from_logits=False: D уже возвращает вероятности (sigmoid в D).
# ----------------------------------------------------------------------
bce = tf.keras.losses.BinaryCrossentropy(from_logits=False)

generator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE, beta_1=BETA_1
)
discriminator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE, beta_1=BETA_1
)


# ----------------------------------------------------------------------
# Шаг 2. train_step(real_images): (B,784) -> два скаляра L_D, L_G.
# ----------------------------------------------------------------------
@tf.function
def train_step(real_images: tf.Tensor):
    batch_size = tf.shape(real_images)[0]

    # ================= Фаза D =================
    # z_D — свой шум для фазы D.
    z_D = tf.random.normal((batch_size, NOISE_DIM))

    with tf.GradientTape() as d_tape:
        # G(z_D) не должен двигать θ_G в фазе D:
        # stop_gradient стоит ТОЛЬКО здесь.
        fake_images = generator(z_D, training=False)
        fake_images = tf.stop_gradient(fake_images)

        d_real = discriminator(real_images, training=True)   # Dropout ON
        d_fake = discriminator(fake_images, training=True)

        d_real_loss = bce(tf.fill(tf.shape(d_real), REAL_LABEL), d_real)
        d_fake_loss = bce(tf.fill(tf.shape(d_fake), FAKE_LABEL), d_fake)
        d_loss = d_real_loss + d_fake_loss

    # Градиенты ТОЛЬКО по θ_D.
    d_grads = d_tape.gradient(d_loss, discriminator.trainable_variables)
    discriminator_optimizer.apply_gradients(
        zip(d_grads, discriminator.trainable_variables)
    )

    # ================= Фаза G =================
    # независимый z_G
    z_G = tf.random.normal((batch_size, NOISE_DIM))

    with tf.GradientTape() as g_tape:
        # training=True: BatchNorm G в режиме обучения.
        fake_images = generator(z_G, training=True)

        # training=False выключает Dropout D, но НЕ обрывает
        # производную к пикселям: граф идёт L_G -> D -> G.
        d_on_fake = discriminator(fake_images, training=False)

        # Генератор хочет, чтобы D сказал «настоящее» = 1.
        g_loss = bce(tf.fill(tf.shape(d_on_fake), GENERATOR_TARGET), d_on_fake)

    # Градиенты ТОЛЬКО по θ_G (θ_D в этом tape нет trainable-участия,
    # потому что d_tape уже применён, а в g_tape мы не обновляем D).
    g_grads = g_tape.gradient(g_loss, generator.trainable_variables)
    generator_optimizer.apply_gradients(
        zip(g_grads, generator.trainable_variables)
    )

    return d_loss, g_loss


# ----------------------------------------------------------------------
# Шаг 3. Контроль: каждый оптимизатор получает только свои переменные,
#         stop_gradient — только в фазе D, градиент G идёт через D.
# ----------------------------------------------------------------------
if __name__ == '__main__':
    # небольшой реальный пакет
    (x_full, _), _ = tf.keras.datasets.fashion_mnist.load_data()
    x_real = x_full[:BATCH_SIZE].reshape(-1, IMAGE_DIM).astype('float32')
    x_real = (x_real / 127.5) - 1.0
    x_real = tf.convert_to_tensor(x_real)

    # 1) train_step исполняется и возвращает два конечных скаляра
    L_D, L_G = train_step(x_real)
    assert np.isfinite(L_D.numpy()) and np.isfinite(L_G.numpy()), (L_D, L_G)
    print(f'L_D={L_D.numpy():.4f}  L_G={L_G.numpy():.4f}')

    # 2) множества переменных не пересекаются
    g_vars = set(id(v) for v in generator.trainable_variables)
    d_vars = set(id(v) for v in discriminator.trainable_variables)
    assert g_vars.isdisjoint(d_vars), 'θ_G and θ_D share variables!'
    print(f'θ_G: {len(g_vars)} vars, θ_D: {len(d_vars)} vars, disjoint=True')

    # 3) stop_gradient — только в фазе D.
    #    В фазе D градиент к θ_G должен отсутствовать:
    z_D = tf.random.normal((BATCH_SIZE, NOISE_DIM))
    with tf.GradientTape() as tape:
        f = generator(z_D, training=False)
        f_stop = tf.stop_gradient(f)
        d_out = discriminator(f_stop, training=True)
        loss_d = bce(tf.ones_like(d_out), d_out)
    g_grads_from_d = tape.gradient(loss_d, generator.trainable_variables)
    assert all(g is None for g in g_grads_from_d), \
        'phase D must NOT produce θ_G gradients (stop_gradient missing?)'
    print('phase D: θ_G grads are all None  (stop_gradient OK)')

    #    В фазе G градиент L_G -> D -> G должен существовать:
    z_G = tf.random.normal((BATCH_SIZE, NOISE_DIM))
    with tf.GradientTape() as tape:
        fake = generator(z_G, training=True)
        d_on_fake = discriminator(fake, training=False)
        loss_g = bce(tf.ones_like(d_on_fake), d_on_fake)
    g_grads_from_g = tape.gradient(loss_g, generator.trainable_variables)
    n_not_none = sum(g is not None for g in g_grads_from_g)
    assert n_not_none > 0, 'phase G must produce θ_G gradients through D'
    print(f'phase G: θ_G grads present for {n_not_none}/{len(g_grads_from_g)} vars  '
          f'(L_G -> D -> G path OK)')

    # 4) после двух шагов состояния Adam G и D не совпадают
    g_iter = int(generator_optimizer.iterations.numpy())
    d_iter = int(discriminator_optimizer.iterations.numpy())
    assert g_iter >= 1 and d_iter >= 1
    assert generator_optimizer is not discriminator_optimizer
    print(f'Adam steps: G={g_iter}, D={d_iter}  (independent optimizers)')

## Этап 6. Эксперименты, сетки и отчёт

**Требуется:** обучите GAN сначала 105 эпох. На фиксированных 100
шумовых векторах сохраните сетки после первой эпохи, далее через
равные интервалы и в конце. Постройте отдельные ряды потерь
дискриминатора и генератора. Затем повторите запуск для 205 и 400 эпох
с теми же архитектурой, шумом, размером выборки и способом
визуализации.

**Проверка:** все значения потерь конечны; формы не меняются;
изображения становятся разнообразнее или обнаруживаются и
объясняются признаки неустойчивости. В отчёте сравните три режима, не
объявляя меньшую потерю одной стороны безусловной победой GAN.

Фиксированный шум — одинаковый набор заданий для художника. Показывая
его после нескольких эпох, вы видите изменение весов, а не новую
случайную карточку. Для Matplotlib значения возвращаются из `[-1,1]`
в `[0,1]`: модель при этом не меняется. Кривые потерь читайте вместе с
сетками, потому что GAN — соревнование, а не одна убывающая ошибка.

In [ ]:
from __future__ import annotations

import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

# ----------------------------------------------------------------------
# Конфигурация (Этап 1). Меняется ТОЛЬКО EPOCHS между прогонами.
# ----------------------------------------------------------------------
SEED = 42
MAX_EXAMPLES = 10_000
IMAGE_HEIGHT = 28
IMAGE_WIDTH = 28
IMAGE_DIM = IMAGE_HEIGHT * IMAGE_WIDTH
NOISE_DIM = 100
BATCH_SIZE = 128
EPOCH_SCHEDULE = (105, 205, 400)   # меняем только это
LEARNING_RATE = 2e-4
BETA_1 = 0.5
REAL_LABEL = 0.9
FAKE_LABEL = 0.0
GENERATOR_TARGET = 1.0
SAVE_EVERY = 20
DROPOUT_RATE = 0.3

artifact_dir = Path('artifacts/fashion_mnist_dense_gan')
artifact_dir.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------------------
# Данные (Этап 2)
# ----------------------------------------------------------------------
def load_dataset():
    (x_full, _), _ = tf.keras.datasets.fashion_mnist.load_data()
    x = x_full[:MAX_EXAMPLES].astype('float32')
    x = (x / 127.5) - 1.0
    x = x.reshape(-1, IMAGE_DIM)
    assert x.shape == (MAX_EXAMPLES, IMAGE_DIM)
    assert np.isfinite(x).all() and x.min() >= -1.0 and x.max() <= 1.0
    ds = (
        tf.data.Dataset.from_tensor_slices(x)
        .shuffle(len(x), seed=SEED, reshuffle_each_iteration=True)
        .batch(BATCH_SIZE, drop_remainder=True)
        .prefetch(tf.data.AUTOTUNE)
    )
    return ds, len(x) // BATCH_SIZE


# ----------------------------------------------------------------------
# Модели и train_step (Этапы 3–5)
# ----------------------------------------------------------------------
def make_initializer():
    return tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02, seed=SEED)


def make_generator():
    init = make_initializer()
    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(NOISE_DIM,)),
            tf.keras.layers.Dense(256, kernel_initializer=init, use_bias=False),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dense(512, kernel_initializer=init, use_bias=False),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dense(1024, kernel_initializer=init, use_bias=False),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dense(IMAGE_DIM, kernel_initializer=init),
            tf.keras.layers.Activation('tanh'),
        ],
        name='generator',
    )


def make_discriminator():
    init = make_initializer()
    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(IMAGE_DIM,)),
            tf.keras.layers.Dense(1024, kernel_initializer=init),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dropout(DROPOUT_RATE),
            tf.keras.layers.Dense(512, kernel_initializer=init),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dropout(DROPOUT_RATE),
            tf.keras.layers.Dense(256, kernel_initializer=init),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Dropout(DROPOUT_RATE),
            tf.keras.layers.Dense(1, kernel_initializer=init),
            tf.keras.layers.Activation('sigmoid'),
        ],
        name='discriminator',
    )


def build_train_step(generator, discriminator, g_opt, d_opt, bce):
    @tf.function
    def train_step(real_images):
        batch_size = tf.shape(real_images)[0]

        # --- фаза D ---
        z_D = tf.random.normal((batch_size, NOISE_DIM))
        with tf.GradientTape() as d_tape:
            fake = tf.stop_gradient(generator(z_D, training=False))
            d_real = discriminator(real_images, training=True)
            d_fake = discriminator(fake, training=True)
            d_loss = (
                bce(tf.fill(tf.shape(d_real), REAL_LABEL), d_real)
                + bce(tf.fill(tf.shape(d_fake), FAKE_LABEL), d_fake)
            )
        d_grads = d_tape.gradient(d_loss, discriminator.trainable_variables)
        d_opt.apply_gradients(zip(d_grads, discriminator.trainable_variables))

        # --- фаза G ---
        z_G = tf.random.normal((batch_size, NOISE_DIM))
        with tf.GradientTape() as g_tape:
            fake = generator(z_G, training=True)
            d_on_fake = discriminator(fake, training=False)
            g_loss = bce(tf.fill(tf.shape(d_on_fake), GENERATOR_TARGET), d_on_fake)
        g_grads = g_tape.gradient(g_loss, generator.trainable_variables)
        g_opt.apply_gradients(zip(g_grads, generator.trainable_variables))

        return d_loss, g_loss

    return train_step


# ----------------------------------------------------------------------
# Снимок сетки сэмплов (Matplotlib/NumPy — только вне tf.function)
# ----------------------------------------------------------------------
def save_sample_grid(generator, fixed_noise, path, step_label):
    fake = generator(fixed_noise, training=False)          # (100, 784)
    fake = (fake.numpy() + 1.0) / 2.0                      # [-1,1] -> [0,1]
    fake = fake.reshape(-1, IMAGE_HEIGHT, IMAGE_WIDTH)     # (100, 28, 28)

    fig, axes = plt.subplots(10, 10, figsize=(10, 10))
    for i, ax in enumerate(axes.flat):
        ax.imshow(fake[i], cmap='gray', vmin=0, vmax=1)    # vmin/vmax явно
        ax.axis('off')
    fig.suptitle(f'G samples — {step_label}', fontsize=14)
    fig.tight_layout()
    fig.savefig(path, dpi=80)
    plt.close(fig)


def plot_losses(history, path, title):
    epochs = np.arange(1, len(history['d_loss']) + 1)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(epochs, history['d_loss'], label='L_D')
    ax.plot(epochs, history['g_loss'], label='L_G')
    ax.set_xlabel('epoch')
    ax.set_ylabel('loss')
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(path, dpi=100)
    plt.close(fig)


# ----------------------------------------------------------------------
# Один прогон: меняется ТОЛЬКО число эпох.
# ----------------------------------------------------------------------
def run_experiment(n_epochs: int):
    tf.keras.utils.set_random_seed(SEED)                  # сброс θ_G, θ_D, шумов

    ds, steps_per_epoch = load_dataset()
    generator = make_generator()
    discriminator = make_discriminator()

    bce = tf.keras.losses.BinaryCrossentropy(from_logits=False)
    g_opt = tf.keras.optimizers.Adam(LEARNING_RATE, beta_1=BETA_1)
    d_opt = tf.keras.optimizers.Adam(LEARNING_RATE, beta_1=BETA_1)

    train_step = build_train_step(generator, discriminator, g_opt, d_opt, bce)

    # fixed_noise создаётся ОДИН раз до цикла, чтобы снимки были сопоставимы
    fixed_noise = tf.random.normal((100, NOISE_DIM))

    run_dir = artifact_dir / f'epochs_{n_epochs}'
    run_dir.mkdir(parents=True, exist_ok=True)

    history = {'d_loss': [], 'g_loss': []}
    t0 = time.time()
    for epoch in range(1, n_epochs + 1):
        d_vals, g_vals = [], []
        for real_batch in ds:
            d_loss, g_loss = train_step(real_batch)
            # скаляры -> числа ТОЛЬКО после выхода из tf.function
            d_vals.append(float(d_loss.numpy()))
            g_vals.append(float(g_loss.numpy()))
        history['d_loss'].append(float(np.mean(d_vals)))
        history['g_loss'].append(float(np.mean(g_vals)))

        if epoch % SAVE_EVERY == 0 or epoch == 1:
            save_sample_grid(
                generator, fixed_noise,
                run_dir / f'samples_epoch_{epoch:03d}.png',
                f'epoch {epoch}/{n_epochs}',
            )
            print(f'[{n_epochs}] epoch {epoch:3d}  '
                  f'L_D={history["d_loss"][-1]:.4f}  '
                  f'L_G={history["g_loss"][-1]:.4f}')

    elapsed = time.time() - t0
    plot_losses(history, run_dir / 'losses.png',
                f'GAN losses — {n_epochs} epochs')

    # сохраняем финальный снимок и историю
    save_sample_grid(generator, fixed_noise,
                     run_dir / 'samples_final.png', f'final ({n_epochs} ep)')
    (run_dir / 'history.json').write_text(json.dumps(history))

    return {
        'epochs': n_epochs,
        'steps_per_epoch': steps_per_epoch,
        'final_d_loss': history['d_loss'][-1],
        'final_g_loss': history['g_loss'][-1],
        'seconds': round(elapsed, 1),
        'run_dir': str(run_dir),
    }


# ----------------------------------------------------------------------
# Сравнение 105 / 205 / 400: меняется ТОЛЬКО EPOCHS
# ----------------------------------------------------------------------
if __name__ == '__main__':
    results = [run_experiment(n) for n in EPOCH_SCHEDULE]

    summary_path = artifact_dir / 'comparison.json'
    summary_path.write_text(json.dumps(results, indent=2))

    # сводный график финальных лоссов по длительностям
    fig, ax = plt.subplots(figsize=(7, 4))
    xs = [r['epochs'] for r in results]
    ax.plot(xs, [r['final_d_loss'] for r in results], 'o-', label='final L_D')
    ax.plot(xs, [r['final_g_loss'] for r in results], 's-', label='final L_G')
    ax.set_xlabel('epochs')
    ax.set_ylabel('final loss')
    ax.set_title('105 vs 205 vs 400 epochs (rest fixed)')
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(artifact_dir / 'comparison.png', dpi=100)
    plt.close(fig)

    for r in results:
        print(f"{r['epochs']:3d} ep | steps/ep={r['steps_per_epoch']} "
              f"| final L_D={r['final_d_loss']:.4f} "
              f"L_G={r['final_g_loss']:.4f} | {r['seconds']}s")

## Этап 7. Итоговая самопроверка

- Использован именно Fashion-MNIST, а не MNIST из примера.
- Шум имеет форму $(B,100)$; реальные и синтетические изображения —
  $(B,784)$; ответ дискриминатора — $(B,1)$.
- Реальные значения нормированы к $[-1,1]$, как и выход tanh
  генератора.
- Метки классов одежды не входят в BinaryCrossentropy GAN.
- Генератор и дискриминатор обновляются отдельными оптимизаторами.
- Сетки и две истории потерь сохранены для всех трёх экспериментов.